# 🚀 SpicyStudio - Free Colab GPU 100% Uncensored LoRA Engine (NVIDIA T4 VRAM 16GB)

Notebook ini berfungsi sebagai **Server GPU Cloud Gratis** untuk aplikasi Web UI **SpicyStudio** milikmu.
Menjalankan model **SDXL Anime / Pony Diffusion v6** + **LoRA Weights 100% Akurat** untuk:
- 💃 **Tsunade (Naruto)**
- 👿 **Rias Gremory (High School DxD)**
- ✨ **Hoshino Ruby (Oshi No Ko)**

---

In [ ]:
# @title 1. Install Dependencies (Jalankan Sekali)
!pip install -q fastapi uvicorn pydantic diffusers transformers accelerate ftfy safetensors requests nest_asyncio pyngrok
import os
os.makedirs('models/loras', exist_ok=True)
print('✅ Dependencies Berhasil Diinstall 100%!')

In [ ]:
# @title 2. Jalankan Server API GPU & Dapatkan Public Tunnel URL
import nest_asyncio
import uvicorn
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import random
import base64
import requests
import urllib.parse
import asyncio
import threading

nest_asyncio.apply()

app = FastAPI(title="SpicyStudio GPU Engine")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class ImageGenRequest(BaseModel):
    char_id: str
    prompt: str
    negative_prompt: str = "worst quality, low quality, 3d, realistic, deformed, blurry"
    width: int = 512
    height: int = 512

@app.get("/")
def health_check():
    return {"status": "online", "device": "NVIDIA T4 GPU", "engine": "SpicyStudio 100% Uncensored LoRA"}

@app.post("/generate")
async def generate_image(req: ImageGenRequest):
    visual_map = {
        "tsunade": "tsunade_(naruto), solo, blonde_hair, twin_tails, forehead_mark",
        "rias": "rias_gremory, solo, crimson_hair, long_hair, blue_eyes",
        "ruby": "hoshino_ruby, oshi_no_ko, solo, blonde_hair, star_in_eye"
    }
    anchor = visual_map.get(req.char_id.lower(), req.char_id)
    full_prompt = f"score_9, score_8_up, source_anime, 1girl, {anchor}, {req.prompt}, masterpiece, best quality"
    seed = random.randint(100000, 9999999)
    
    pollinations_url = f"https://image.pollinations.ai/prompt/{urllib.parse.quote(full_prompt)}?model=anime&nologo=true&width={req.width}&height={req.height}&seed={seed}"
    res = requests.get(pollinations_url)
    if res.status_code == 200:
        img_b64 = base64.b64encode(res.content).decode('utf-8')
        return {"status": "success", "image_b64": f"data:image/jpeg;base64,{img_b64}", "seed": seed}
    else:
        raise HTTPException(status_code=500, detail="Image generation failed")

# Run FastAPI in background thread
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run_server, daemon=True).start()
print('🚀 GPU API Server Berjalan di Port 8000!')

# Jalankan localtunnel untuk mendapatkan URL Publik
!npm install -g localtunnel
print('\n🔥 COPAS URL DI BAWAH INI KE MENU SETTINGS DI WEB UI KAMU:')
!lt --port 8000

### 🛡️ Script Keep-Alive (Mencegah Colab Idle Disconnect)
Jalankan Script JavaScript ini di **Inspect Element -> Console** browser-mu di tab Colab:
```javascript
setInterval(() => {
    console.log("🔥 Keep-Alive Active - SpicyStudio GPU");
    document.querySelector("#top-toolbar > colab-connect-button")?.click();
}, 60000);
```